In [1]:
!pip install mysql-connector-python
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import mysql.connector
from mysql.connector import Error
import wbgapi as wb
import os


def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

In [110]:
def preprocess_data(df):
    """
    Preprocesses data for each country by handling NaN values and organizing data for analysis.
    
    This function processes the input DataFrame to prepare data for each country:
    1. Filters the data for each country
    2. Sorts the data by year
    3. Handles NaN values by either skipping countries with more than one NaN or interpolating single NaN values
    4. Creates a list of dictionaries containing the country name, original data, and cleaned values
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    
    Returns:
    list: A list of dictionaries, where each dictionary contains:
        - "country": The country name
        - "data": The original country data sorted by year
        - "values": Cleaned numeric values with NaNs handled
    """
    countries = df["country"].unique()
    preprocessed_data = []

    for country in countries:
        country_data = df[df["country"] == country].sort_values("year")
        values = country_data["value"].values.copy()

        # Handle NaN values
        nan_indices = np.isnan(values)
        if np.sum(nan_indices) > 1:
            continue

        if np.sum(nan_indices) == 1:
            nan_pos = np.where(nan_indices)[0][0]
            if nan_pos == 0:
                values[nan_pos] = values[nan_pos + 1]
            elif nan_pos == len(values) - 1:
                values[nan_pos] = values[nan_pos - 1]
            else:
                values[nan_pos] = (values[nan_pos - 1] + values[nan_pos + 1]) / 2

        preprocessed_data.append({
            "country": country,
            "data": country_data,
            "values": values
        })

    return preprocessed_data

def calculate_total_comparisons(df, min_length=3, max_length=5):
    """
    Calculate the total number of segment comparisons for progress tracking.
    
    This function computes the total number of comparisons that would be made 
    during the pattern extraction process. It considers all possible segment 
    lengths within the specified range and all country pairs.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    min_length (int): Minimum segment length to consider
    max_length (int): Maximum segment length to consider
    
    Returns:
    int: Total number of comparisons that would be performed
    """
    preprocessed_data = preprocess_data(df)
    total_comparisons = 0

    for i, country_a in enumerate(preprocessed_data):
        values_a = country_a["values"]

        for segment_length in range(min_length, max_length + 1):
            if len(values_a) < segment_length:
                continue

            num_segments_a = len(values_a) - segment_length + 1
            num_segments_b = sum(
                len(country_b["values"]) - segment_length + 1
                for country_b in preprocessed_data[i+1:]
                if len(country_b["values"]) >= segment_length
            )
            total_comparisons += num_segments_a * num_segments_b

    return total_comparisons

#### Show Parquet as DF

In [109]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\PV.EST\Angola.parquet")
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,6,2,3,1996,1998,0.552982
1,6,2,3,1996,2005,0.572103
2,6,2,3,1996,2010,0.963827
3,6,2,3,1996,2015,0.648659
4,6,2,3,1996,2019,0.710162
...,...,...,...,...,...,...
175789,6,197,24,1998,1998,0.854341
175790,6,219,24,1998,1996,0.767484
175791,6,219,24,1998,1998,0.676904
175792,6,232,24,1998,1996,0.639067


#### Shrinking

In [18]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL\SP.POP.TOTL.parquet")
output_path = r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet"

# Select 5 countries to keep
target_countries = ["United States", "United Kingdom", "Germany", "France", "Canada", "Australia", "Japan", "India", "China", "Brazil", "Mexico", "South Africa", "Nigeria", "Egypt", "Turkiye", "Russia", "Italy", "Spain", "Argentina", "South Korea", "Indonesia", "Pakistan", "Bangladesh", "Thailand", "Philippines", "Vietnam", "Kenya", "Tanzania", "Uganda", "Ghana", "Senegal", "Morocco", "Algeria", "Israel", "Saudi Arabia", "Iran", "Iraq", "Afghanistan", "Poland", "Hungary", "Czech", "Greece", "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", "Finland", "Switzerland", "Austria"]

# Filter the DataFrame
filtered_df = df[df["country"].isin(target_countries)]

filtered_df.to_parquet(output_path, index = False, compression = "snappy")
print(f"Shrinked Parquet file saved to {output_path}")
filtered_df

Shrinked Parquet file saved to /workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet


,country,year,value
128,Afghanistan,1960,9035043.0
129,Afghanistan,1961,9214083.0
130,Afghanistan,1962,9404406.0
131,Afghanistan,1963,9604487.0
132,Afghanistan,1964,9814318.0
...,...,...,...
16891,South Africa,2019,59587885.0
16892,South Africa,2020,60562381.0
16893,South Africa,2021,61502603.0
16894,South Africa,2022,62378410.0


In [5]:
from typing import Sequence


def process_sequence(seq: Sequence[int]) -> None:
    for item in seq:
        print(item)

# Populate a list and pass it to the function
my_sequence = [1, 2, 3]
process_sequence(my_sequence)

1
2
3


#### Calculating Comaprisons

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\NY.GDP.MKTP.KD.ZG\NY.GDP.MKTP.KD.ZG.parquet")

# Example usage:
total = calculate_total_comparisons(df)
print(f"Total comparisons: {total}")

Total comparisons: 60960900


#### Size Saving Comparison

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked_sdcis.parquet")
print(df.head())

   country_a_id  country_b_id  length  start_year_a  start_year_b  correlation
0            45            81       3          1960          1977     0.945475
1            45            81       3          1961          1960     0.970550
2            45            81       3          1961          1961     0.964687
3            45            81       3          1961          1962     0.948679
4            45            81       3          1961          1963     0.965124


In [15]:
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,45,81,3,1960,1977,0.945475
1,45,81,3,1961,1960,0.970550
2,45,81,3,1961,1961,0.964687
3,45,81,3,1961,1962,0.948679
4,45,81,3,1961,1963,0.965124
...,...,...,...,...,...,...
82764,110,236,5,2005,2015,0.958875
82765,110,236,5,2005,2017,0.943587
82766,110,236,5,2005,2018,0.978710
82767,110,236,5,2005,2019,0.901636


In [13]:
df

,country_a,country_b,length,start_year_a,start_year_b,correlation,data_segment_a,data_segment_b
0,China,Germany,3,1960,1977,0.945475,"[667070000.0, 660330000.0, 665770000.0]","[78159814.0, 78091820.0, 78126350.0]"
1,China,Germany,3,1961,1960,0.970550,"[660330000.0, 665770000.0, 682335000.0]","[72814900.0, 73377632.0, 74025784.0]"
2,China,Germany,3,1961,1961,0.964687,"[660330000.0, 665770000.0, 682335000.0]","[73377632.0, 74025784.0, 74714353.0]"
3,China,Germany,3,1961,1962,0.948679,"[660330000.0, 665770000.0, 682335000.0]","[74025784.0, 74714353.0, 75318337.0]"
4,China,Germany,3,1961,1963,0.965124,"[660330000.0, 665770000.0, 682335000.0]","[74714353.0, 75318337.0, 75963695.0]"
...,...,...,...,...,...,...,...,...
82764,Japan,United States,5,2005,2015,0.958875,"[127773000.0, 127854000.0, 128001000.0, 128063...","[320738994.0, 323071755.0, 325122128.0, 326838..."
82765,Japan,United States,5,2005,2017,0.943587,"[127773000.0, 127854000.0, 128001000.0, 128063...","[325122128.0, 326838199.0, 328329953.0, 331526..."
82766,Japan,United States,5,2005,2018,0.978710,"[127773000.0, 127854000.0, 128001000.0, 128063...","[326838199.0, 328329953.0, 331526933.0, 332048..."
82767,Japan,United States,5,2005,2019,0.901636,"[127773000.0, 127854000.0, 128001000.0, 128063...","[328329953.0, 331526933.0, 332048977.0, 333271..."


#### Comparing Parquet with MySQL

##### MySQL Setup

In [ ]:
# def create_mysql_user():
#     try:
#         # Connect as root or another admin user
#         connection = mysql.connector.connect(
#             host='localhost',
#             user='root',
#             password='<MYSQL_ROOT_PASSWORD>'
#         )

#         if connection.is_connected():
#             cursor = connection.cursor()
#             cursor.execute("CREATE USER 'qgi_user'@'localhost' IDENTIFIED BY '<MYSQL_PASSWORD>")
#             cursor.execute("GRANT ALL PRIVILEGES ON sdci_pattern_db.* TO 'qgi_user'@'localhost';")
#             cursor.execute("FLUSH PRIVILEGES;")
#             connection.commit()
#             print("MySQL user created successfully")

#     except Error as e:
#         print(f"Error creating MySQL user: {e}")
#     finally:
#         if connection.is_connected():
#             cursor.close()
#             connection.close()

# create_mysql_user()

MySQL user created successfully


In [ ]:
# def grant_privileges_to_user():
#     try:
#         # Connect as root or another admin user
#         admin_connection = mysql.connector.connect(
#             host='localhost',
#             user='root',  # Or another admin username
#             password='<MYSQL_ROOT_PASSWORD>'
#         )

#         if admin_connection.is_connected():
#             admin_cursor = admin_connection.cursor()
            
#             # Grant necessary privileges to 'amin' user
#             admin_cursor.execute("GRANT CREATE ON *.* TO 'qgi_user'@'localhost';")
#             admin_cursor.execute("GRANT DROP ON *.* TO 'qgi_user'@'localhost';")
#             admin_cursor.execute("FLUSH PRIVILEGES;")
            
#             # Commit the changes
#             admin_connection.commit()
#             print("Privileges granted successfully to user 'amin'")

#     except Error as e:
#         print(f"Error granting privileges: {e}")
#     finally:
#         if admin_connection.is_connected():
#             admin_cursor.close()
#             admin_connection.close()
#             print("Admin MySQL connection is closed")

# # Run the function to grant privileges
# grant_privileges_to_user()

Privileges granted successfully to user 'amin'
Admin MySQL connection is closed


In [ ]:
# def create_database():
#     """Create a new MySQL database and required tables"""
#     try:
#         # Connect to MySQL server
#         connection = mysql.connector.connect(
#             host='localhost',
#             user='amin',
#             password='<MYSQL_PASSWORD>'
#         )
        
#         if connection.is_connected():
#             db_info = connection.get_server_info()
#             print(f"Connected to MySQL Server version {db_info}")
            
#             cursor = connection.cursor()
            
#             # Create database
#             db_name = "sdci_database"
#             cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
#             print(f"Database '{db_name}' created successfully")
            
#             # Use the new database
#             cursor.execute(f"USE {db_name}")
            
#             # Create SDCIs table
#             create_table_query = """
#             CREATE TABLE IF NOT EXISTS sdcis (
#                 id INT AUTO_INCREMENT PRIMARY KEY,
#                 country_a_id INT NOT NULL,
#                 country_b_id INT NOT NULL,
#                 length INT NOT NULL,
#                 start_year_a INT NOT NULL,
#                 start_year_b INT NOT NULL,
#                 correlation FLOAT NOT NULL,
#                 indicator_id VARCHAR(255) NOT NULL,
#                 created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
#             ) ENGINE=InnoDB;
#             """
#             cursor.execute(create_table_query)
#             print("Table 'sdcis' created successfully")
            
#             # Commit the transaction
#             connection.commit()

#     except Error as e:
#         print(f"Error while creating database: {e}")
#     finally:
#         if connection.is_connected():
#             cursor.close()
#             connection.close()
#             print("MySQL connection is closed")

# create_database()

Connected to MySQL Server version 8.0.41
Database 'sdci_database' created successfully
Table 'sdcis' created successfully
MySQL connection is closed


##### Comparison

In [ ]:
def get_parquet_file_size(file_path):
    """Get the size of a Parquet file in MB"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist")
    return os.path.getsize(file_path) / (1024 * 1024)  # Convert to MB

def get_mysql_table_size(host, user, password, database, table_name):
    """Get the size of a MySQL table in MB"""
    try:
        connection = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        cursor = connection.cursor()
        
        query = f"""
        SELECT 
            TABLE_NAME, 
            ROUND(SUM(DATA_LENGTH + INDEX_LENGTH) / 1024 / 1024, 2) AS table_size_mb 
        FROM 
            information_schema.TABLES 
        WHERE 
            TABLE_SCHEMA = '{database}' 
            AND TABLE_NAME = '{table_name}'
        GROUP BY 
            TABLE_NAME;
        """
        cursor.execute(query)
        result = cursor.fetchone()
        if result:
            return result[1]
        else:
            return 0
        
    except Error as e:
        print(f"Error: {e}")
        return None
    finally:
        if connection.is_connected():
            cursor.close()
            connection.close()

# Example usage
parquet_file_path = r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\SP.POP.TOTL\SP.POP.TOTL_sdcis.parquet"
mysql_host = "localhost"
mysql_user = "amin"
mysql_password = "<MYSQL_PASSWORD>"
mysql_database = "sdci_database"
mysql_table = "sdcis"

# Get Parquet file size
try:
    parquet_size = get_parquet_file_size(parquet_file_path)
    print(f"Parquet file size: {parquet_size:.2f} MB")
except FileNotFoundError as e:
    print(e)

# Get MySQL table size
mysql_size = get_mysql_table_size(
    mysql_host, 
    mysql_user, 
    mysql_password, 
    mysql_database, 
    mysql_table
)
if mysql_size is not None:
    print(f"MySQL table size: {mysql_size} MB")

Parquet file size: 21.08 MB
Error: 1045 (28000): Access denied for user 'qgi_user'@'localhost' (using password: YES)


UnboundLocalError: cannot access local variable 'connection' where it is not associated with a value

##### Testing WB Extractor

- PV.EST:             25,016,070 - Political Stability
- NY.GDP.MKTP.CD:     68,162,325 - GDP (current US$)
- SP.DYN.LE00.IN:    217,120,200 - Life expectancy at birth, total (years)
- SE.PRM.ENRR:           936,600 - School enrollment, primary (% gross)
- SL.EMP.TOTL.SP.ZS:  45,988,040 - Employment to population ratio, 15+, total (%) (modeled ILO estimate)
- SL.UEM.TOTL.ZS:     45,988,040 - Unemployment, total (% of total labor force) (modeled ILO estimate)
- NY.GDP.PCAP.CD:     68,162,325 - GDP per capita (current US$)
- MS.MIL.MPRT.KD:      7,033,950 - Arms imports (SIPRI trend indicator values)

In [129]:
class Indicator:
    def __init__(self):
        self.id                   = ""
        self.data                 = pd.DataFrame()
        self.indicator_name       = ""
        self.concept              = ""
        self.license_type         = ""
        self.license_url          = ""
        self.long_definition      = ""
        self.othernotes           = ""
        self.periodicity          = ""
        self.related_source_links = ""
        self.source               = ""
        self.topic                = ""

def get_indicator(indicator_id):
    """
    Example output:
    id:                   SP.POP.TOTL
    data columns:         ['YR1960', 'YR1961', 'YR1962', 'YR1963', 'YR1964', 'YR1965', 'YR1966',
                           'YR1967', 'YR1968', 'YR1969', 'YR1970', 'YR1971', 'YR1972', 'YR1973',
                           'YR1974', 'YR1975', 'YR1976', 'YR1977', 'YR1978', 'YR1979', 'YR1980',
                           'YR1981', 'YR1982', 'YR1983', 'YR1984', 'YR1985', 'YR1986', 'YR1987',
                           'YR1988', 'YR1989', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994',
                           'YR1995', 'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001',
                           'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008',
                           'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015',
                           'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022',
                           'YR2023']
    data rows:            266
    name:                 Population, total
    concept:              Series
    license_type:         CC BY-4.0
    license_url:          https://datacatalog.worldbank.org/public-licenses#cc-by
    long_definition:      Total population is based on the de facto definition of population, 
                          which counts all residents regardless of legal status or citizenship. 
                          The values shown are midyear estimates.
    othernotes:           
    periodicity:          Annual
    related_source_links: 
    source:               (1) United Nations Population Division. World Population Prospects:
                          2024 Revision; (2) Statistical databases and publications from national
                          statistical offices; (3) Eurostat: Demographic Statistics;
                          (4) United Nations Statistics Division. Population and Vital Statistics
                          Report (various years).
    topic:                Health: Population: Structure
    """
    indicator          = Indicator()
    try:
        indicator_metadata = wb.series.metadata.get(indicator_id)
    except Exception as e:
        print("Error caught when extracting indicator metadata: ", str(e))
    indicator.id       = indicator_id
    indicator.data     = wb.data.DataFrame(indicator_id)

    if "IndicatorName" in list(indicator_metadata.metadata.keys()):
        indicator.name = indicator_metadata.metadata["IndicatorName"]
    else:
        indicator.name = ""

    indicator.concept = indicator_metadata.concept

    if "License_Type" in list(indicator_metadata.metadata.keys()):
        indicator.license_type = indicator_metadata.metadata["License_Type"]
    else:
        indicator.license_type = ""

    if "License_URL" in list(indicator_metadata.metadata.keys()):
        indicator.license_url = indicator_metadata.metadata["License_URL"]
    else:
        indicator.license_url = ""

    if "Longdefinition" in list(indicator_metadata.metadata.keys()):
        indicator.long_definition = indicator_metadata.metadata["Longdefinition"]
    else:
        indicator.long_definition = ""

    if "Othernotes" in list(indicator_metadata.metadata.keys()):
        indicator.othernotes = indicator_metadata.metadata["Othernotes"]
    else:
        indicator.othernotes = ""

    if "Periodicity" in list(indicator_metadata.metadata.keys()):
        indicator.periodicity = indicator_metadata.metadata["Periodicity"]
    else:
        indicator.periodicity = ""

    if "Relatedsourcelinks" in list(indicator_metadata.metadata.keys()):
        indicator.related_source_links = indicator_metadata.metadata["Relatedsourcelinks"]
    else:
        indicator.related_source_links = ""

    if "Source" in list(indicator_metadata.metadata.keys()):
        indicator.source = indicator_metadata.metadata["Source"]
    else:
        indicator.source = ""

    if "Topic" in list(indicator_metadata.metadata.keys()):
        indicator.topic = indicator_metadata.metadata["Topic"]
    else:
        indicator.topic = ""

    return indicator

###########################
indicator = get_indicator("PV.EST")

indicator.data.head()

def remove_all_nan_year_columns(df):
    """TODO"""

    for year_col in list(df):
        nan_status = list(set(np.isnan(list(df[year_col]))))
        if len(nan_status) == 1 and str(nan_status[0]) == "True":
            df = df.drop([year_col], axis = 1)

    return df

def pivot_dataframe(df):
    """
    Reads a wide-format CSV file and pivots it into a long format.

    Parameters:
        file_path (str): Path to the input CSV file.

    Returns:
        pd.DataFrame: A DataFrame in long format with columns:
                      - 'economy': The economy code (e.g., ABW, AFE, AFG).
                      - 'year': The year extracted from column names (e.g., 1960, 1961).
                      - 'value': The corresponding value for the economy and year.
    """
    # Melt the DataFrame to convert year columns into rows
    df["economy"] = df.index
    df_long = pd.melt(
        df,
        id_vars=['economy'],  # Keep 'economy' as an identifier
        var_name='year',      # New column for years
        value_name='value'    # New column for values
    )

    # Extract the numeric part of the year (remove 'YR' prefix)
    df_long['year'] = df_long['year'].str.extract(r'(\d+)').astype(int)

    # Sort the DataFrame by economy and year for better readability
    df_long = df_long.sort_values(by=['economy', 'year']).reset_index(drop=True)

    return df_long

def rename_columns(df, column_mapping = None):
    if column_mapping is None:
        column_mapping = {
            'economy': 'country',
        }
    return df.rename(columns=column_mapping)

## TODO: figure out a better way to link "countries.csv" with this function,
## rather than reading it multiple times and taking out the "Unnamed" columns.
## TODO centralize paths
def clean_country_names(input_df):
    """
    Cleans the 'country' column of the input DataFrame by replacing codes with their corresponding country names.
    
    Parameters:
        input_df (pd.DataFrame): The input DataFrame with a 'country' column to be cleaned.
    
    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: A DataFrame with the 'country' column replaced by country names.
            - list: A list of unmatched values from the 'country' column.
    """
    # Step 1: Load the reference CSV file
    reference_df = pd.read_csv(r"/workspace/qgi\qgi-4.0\data\countries.csv") # r"/workspaces/qgi-4.0/data/countries.csv"
    reference_df = reference_df.loc[:, ~reference_df.columns.str.contains("Unnamed")]

    # Ensure 'name_1' exists in the reference DataFrame
    if 'name_1' not in reference_df.columns:
        raise KeyError("The column 'name_1' is missing in the reference data.")

    # Step 2: Create a mapping dictionary
    mapping = {}
    for col in reference_df.columns:
        if col != 'name_1':  # Skip the 'name_1' column itself
            temp_mapping = reference_df.dropna(subset=[col]).set_index(col)['name_1'].to_dict()
            mapping.update(temp_mapping)

    # Step 3: Replace values in the 'country' column using the mapping
    unmatched_values = []  # List to store unmatched values
    input_df['country'] = input_df['country'].apply(
        lambda x: mapping.get(x, x)  # Replace if found in mapping, otherwise keep original
    )
    # Identify unmatched values
    unmatched_values = [
        value for value in input_df['country'] if value not in mapping.values() and value not in mapping.keys()
    ]

    return input_df, list(set(unmatched_values))

def delete_rows_by_country(input_df, countries_to_delete):
    """
    Deletes rows from the input DataFrame where the 'country' column matches any value in the given list.

    Parameters:
        input_df (pd.DataFrame): The input DataFrame.
        countries_to_delete (list): A list of country names or codes to delete.

    Returns:
        pd.DataFrame: A DataFrame with the specified rows removed.
    """
    # Filter out rows where the 'country' column matches any value in the list
    filtered_df = input_df[~input_df['country'].isin(countries_to_delete)]
    
    return filtered_df

indicator.data = remove_all_nan_year_columns(indicator.data)
indicator.data = pivot_dataframe(indicator.data)
indicator.data = rename_columns(indicator.data)
indicator.data, unmatched_countries = clean_country_names(indicator.data)
indicator.data = delete_rows_by_country(indicator.data, unmatched_countries)

print(calculate_total_comparisons(indicator.data))

indicator.data

25016070


,country,year,value
0,Aruba,1996,NaN
1,Aruba,1998,NaN
2,Aruba,2000,NaN
3,Aruba,2002,NaN
4,Aruba,2003,NaN
...,...,...,...
6645,Zimbabwe,2019,-0.943303
6646,Zimbabwe,2020,-1.052743
6647,Zimbabwe,2021,-0.954443
6648,Zimbabwe,2022,-0.894974


In [ ]:
class Indicator:
    def __init__(self):
        self.id                   = ""
        self.data                 = pd.DataFrame()
        self.indicator_name       = ""
        self.concept              = ""
        self.license_type         = ""
        self.license_url          = ""
        self.long_definition      = ""
        self.othernotes           = ""
        self.periodicity          = ""
        self.related_source_links = ""
        self.source               = ""
        self.topic                = ""

def get_indicator(indicator_id):
    """
    Example output:
    id:                   SP.POP.TOTL
    data columns:         ['YR1960', 'YR1961', 'YR1962', 'YR1963', 'YR1964', 'YR1965', 'YR1966',
                           'YR1967', 'YR1968', 'YR1969', 'YR1970', 'YR1971', 'YR1972', 'YR1973',
                           'YR1974', 'YR1975', 'YR1976', 'YR1977', 'YR1978', 'YR1979', 'YR1980',
                           'YR1981', 'YR1982', 'YR1983', 'YR1984', 'YR1985', 'YR1986', 'YR1987',
                           'YR1988', 'YR1989', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994',
                           'YR1995', 'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001',
                           'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008',
                           'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015',
                           'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022',
                           'YR2023']
    data rows:            266
    name:                 Population, total
    concept:              Series
    license_type:         CC BY-4.0
    license_url:          https://datacatalog.worldbank.org/public-licenses#cc-by
    long_definition:      Total population is based on the de facto definition of population, 
                          which counts all residents regardless of legal status or citizenship. 
                          The values shown are midyear estimates.
    othernotes:           
    periodicity:          Annual
    related_source_links: 
    source:               (1) United Nations Population Division. World Population Prospects:
                          2024 Revision; (2) Statistical databases and publications from national
                          statistical offices; (3) Eurostat: Demographic Statistics;
                          (4) United Nations Statistics Division. Population and Vital Statistics
                          Report (various years).
    topic:                Health: Population: Structure
    """
    indicator          = Indicator()
    indicator_metadata = wb.series.metadata.get(indicator_id)
    indicator.id       = indicator_id
    indicator.data     = wb.data.DataFrame(indicator_id)

    if "IndicatorName" in list(indicator_metadata.metadata.keys()):
        indicator.name = indicator_metadata.metadata["IndicatorName"]
    else:
        indicator.name = ""

    indicator.concept = indicator_metadata.concept

    if "License_Type" in list(indicator_metadata.metadata.keys()):
        indicator.license_type = indicator_metadata.metadata["License_Type"]
    else:
        indicator.license_type = ""

    if "License_URL" in list(indicator_metadata.metadata.keys()):
        indicator.license_url = indicator_metadata.metadata["License_URL"]
    else:
        indicator.license_url = ""

    if "Longdefinition" in list(indicator_metadata.metadata.keys()):
        indicator.long_definition = indicator_metadata.metadata["Longdefinition"]
    else:
        indicator.long_definition = ""

    if "Othernotes" in list(indicator_metadata.metadata.keys()):
        indicator.othernotes = indicator_metadata.metadata["Othernotes"]
    else:
        indicator.othernotes = ""

    if "Periodicity" in list(indicator_metadata.metadata.keys()):
        indicator.periodicity = indicator_metadata.metadata["Periodicity"]
    else:
        indicator.periodicity = ""

    if "Relatedsourcelinks" in list(indicator_metadata.metadata.keys()):
        indicator.related_source_links = indicator_metadata.metadata["Relatedsourcelinks"]
    else:
        indicator.related_source_links = ""

    if "Source" in list(indicator_metadata.metadata.keys()):
        indicator.source = indicator_metadata.metadata["Source"]
    else:
        indicator.source = ""

    if "Topic" in list(indicator_metadata.metadata.keys()):
        indicator.topic = indicator_metadata.metadata["Topic"]
    else:
        indicator.topic = ""

    return indicator

indicator = get_indicator("SI.POV.DDAY")

indicator.data.head()

,YR1960,YR1961,YR1962,YR1963,YR1964,YR1965,YR1966,YR1967,YR1968,YR1969,...,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
economy,,,,,,,,,,,,,,,,,,,,,
ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,31.1,NaN,NaN,NaN,NaN,NaN


# Other

## Step 1: Preprocessing Data

- WB.WWGI.PV.EST (PV.EST) (political stability)
- BX.TRF.PWKR.DT.GD.ZS (Personal remittances, received (% of GDP))
- Inflation, GDP deflator (annual %) (this or Normal inflation?)
- Migrant Stock UNRELIABLE INDICATOR

Reference Sheets:

- [world remitances as a share of gdp - world bank data](https://docs.google.com/spreadsheets/d/1O90WM2FWPPc945OAnqrepDaEDxPQGU3Y/edit?gid=1907481259#gid=1907481259)
- [Political Stability Index](https://docs.google.com/spreadsheets/d/1zywl7lL-T1QykUM_MoAQxoZ_v-uiHv7Sn7_4r8m3OpQ/edit?gid=515249344#gid=515249344)
- [API_NY.GDP.DEFL.KD.ZG_DS2_en_excel_v2_76097](https://docs.google.com/spreadsheets/d/10fYIkUM1wV1yk6S5PINnqK9B8yewwEGD2KLnNbeQR14/edit?gid=332287218#gid=332287218)
- [World Bank Data](https://data.worldbank.org)
- [Tunisia EUDIF](https://diasporafordevelopment.eu/library/tunisia/)
- [Political Stability](https://docs.google.com/spreadsheets/d/1Qd2dRIRRFFs7M_eWX1KiGIYKYS1GR5C86t5APsrZ1CQ/edit?gid=1959161449#gid=1959161449)
- [Inflation, GDP deflator (annual %)](https://docs.google.com/spreadsheets/d/1htc2bg4KFkZ8_mq5TpPklO2xGTZY81lN6dk8DFJtdQ4/edit?gid=455678384#gid=455678384)
- [Personal Remittances, recieved (% of GDP)](https://docs.google.com/spreadsheets/d/1rXJcyuzehn8i_Mbt0z-i2pNh1QPlb1T0EfJjT0G8BMg/edit?gid=1670822621#gid=1670822621)
- [GDP (current US$)](https://docs.google.com/spreadsheets/d/1z4tUfR7sxZ93xsHKZMQVt_q2elTsXBce1QNZElDfB6w/edit?gid=1953930118#gid=1953930118)

In [69]:
!pip install scikit-learn causalml statsmodels
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind
from causalml.match import NearestNeighborMatch
from causalml.propensity import ElasticNetPropensityModel
from causalml.inference.meta import BaseXRegressor
import numpy as np
from statsmodels.stats.weightstats import ttest_ind

inf_path       = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\exports\Inflation, GDP deflator (annual %) - Data.csv"
rem_path       = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\exports\Personal Remittances, recieved (% of GDP) - Data.csv"
pol_path       = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\exports\Political Stability - Data.csv"
gdp_path       = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\exports\GDP (current US$) - Data.csv"
sara_data_path = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\world remitances as a share of gdp - world bank data.xlsx - Data.csv"
res_path       = r"/workspace/qgi\qgi-4.0\notebooks\Sarah\exports\processed_data.csv"

In [71]:
countries_of_interest = [
    'Jordan', 'Belize', 'Gabon', 'Djibouti', 'Maldives', 'Cote d\'Ivoire',
    'Malaysia', 'Congo, Rep', 'Suriname', 'Dominican Republic', 'Eswatini',
    'Rwanda', 'Guinea', 'Cabo Verde', 'Togo', 'Benin', 'Sudan', 'Paraguay',
    'Thailand', 'Malawi', 'Samoa', 'Nepal', 'Senegal', 'Mali', 'Fiji',
    'Pakistan', 'Ecuador', 'Kenya', 'Sierra Leone', 'Cameroon', 'El Salvador',
    'Honduras', 'Nicaragua', 'Guatemala', 'Vanuatu', 'Bangladesh',
    'Guinea-Bissau', 'Cambodia', 'Nigeria', 'Bolivia', 'India', 'Ghana',
    'Mexico', 'Colombia', 'Niger', 'Algeria', 'Ethiopia', 'Lao PDR',
    'Lesotho', 'Philippines', 'Sri Lanka', 'Egypt, Arab Rep.', 'Morocco',
    'Tunisia', 'Madagascar', 'Myanmar', 'Indonesia', 'Brazil',
    'Papua New Guinea', 'Palestine (West Bank and Gaza)'
]


In [72]:
def process_dataset(file_path, indicator_name):
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Select relevant columns
    id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    value_vars = [str(year) for year in range(1960, 2024)]
    
    # Melt the dataframe to long format
    df_melted = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name='year',
        value_name=indicator_name
    )
    
    # Rename columns
    df_melted.rename(columns={'Country Name': 'country'}, inplace=True)
    
    # Filter for countries of interest
    df_melted = df_melted[df_melted['country'].isin(countries_of_interest)]
    
    # Drop rows with missing values for the indicator
    df_melted = df_melted.dropna(subset=[indicator_name])
    
    # Convert year to integer
    df_melted['year'] = df_melted['year'].astype(int)
    
    # Keep only necessary columns
    df_clean = df_melted[['country', 'year', indicator_name]]
    
    return df_clean


In [73]:
# Step 1: Data Preprocessing
# Process each indicator
inflation_df = process_dataset(inf_path, 'inflation_gdp_deflator')
remittances_df = process_dataset(rem_path, 'remittances_gdp')
political_stability_df = process_dataset(pol_path, 'political_stability')
gdp_df = process_dataset(gdp_path, 'gdp')  # Add this line

# Merge all datasets
merged_df = pd.merge(inflation_df, remittances_df, on=['country', 'year'], how='outer')
merged_df = pd.merge(merged_df, political_stability_df, on=['country', 'year'], how='outer')
merged_df = pd.merge(merged_df, gdp_df, on=['country', 'year'], how='outer')  # Add this line

# Handle remaining missing values
merged_df = merged_df.dropna()

# Read the DEP dataframe
deps = pd.read_csv(sara_data_path)
deps = pd.DataFrame({"country": deps["Country Name"], "DEP": deps["DEP"]})

# Merge DEP information
merged_df = pd.merge(merged_df, deps, on='country', how='left')

# Fill missing DEP values with 0
merged_df['DEP'] = merged_df['DEP'].fillna(0).astype(int)

# Saving cleaned data
merged_df.to_csv(res_path, index=False)
merged_df.head()

,country,year,inflation_gdp_deflator,remittances_gdp,political_stability,gdp,DEP
0,Algeria,1996,24.021904,1.874672,-1.783311,4.694155e+10,0
1,Algeria,1998,-3.131089,2.199728,-1.878905,4.818778e+10,0
2,Algeria,2000,22.678009,1.441858,-1.432577,5.479040e+10,0
3,Algeria,2002,1.372856,1.739382,-1.634565,6.151610e+10,0
4,Algeria,2003,8.942462,2.381527,-1.754409,7.348226e+10,0


## Step 2: Treatment, Outcome & Confounders

In [74]:
# Step 2: Define treatment, outcome, and confounders
# First, ensure all confounders are present in the data
required_confounders = ['gdp', 'inflation_gdp_deflator', 'political_stability']

# Check if all confounders are present
missing_confounders = [confounder for confounder in required_confounders if confounder not in merged_df.columns]
if missing_confounders:
    print(f"Missing required confounders: {', '.join(missing_confounders)}")
    print("Please add these variables to the dataset before proceeding.")
else:
    # Define variables
    treatment = merged_df['DEP']
    outcome = merged_df['remittances_gdp']
    confounders = merged_df[required_confounders]

    # Standardize confounders
    scaler = StandardScaler()
    confounders_scaled = scaler.fit_transform(confounders)

## Step 3: Propensity Scores

In [75]:
# Step 3: Calculate propensity scores
# Initialize logistic regression model
ps_model = LogisticRegression()

# Fit the model to the standardized confounders and treatment variable
ps_model.fit(confounders_scaled, treatment)

# Predict probabilities of treatment
propensity_scores = ps_model.predict_proba(confounders_scaled)[:, 1]

# Add propensity scores to the dataframe
merged_df['propensity_score'] = propensity_scores

## Step 4: Propensity Score Matching

In [77]:
# Step 4: Perform propensity score matching
def find_nearest(array, value, caliper):
    """Find the nearest value in an array within a specified caliper."""
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    if np.abs(array[idx] - value) <= caliper:
        return idx
    else:
        return None

def perform_propensity_matching(df, caliper=0.2):
    """Perform propensity score matching and return matched dataframe."""
    # Extract treated and control groups
    treated = df[df['DEP'] == 1]
    control = df[df['DEP'] == 0]
    
    # Get propensity scores
    treated_scores = treated['propensity_score'].values
    control_scores = control['propensity_score'].values
    
    # Find matches
    matched_indices = []
    for i, score in enumerate(treated_scores):
        match_idx = find_nearest(control_scores, score, caliper)
        if match_idx is not None:
            control_index = control.iloc[match_idx].name
            matched_indices.append((treated.index[i], control_index))
    
    # Create matched dataframe
    matched_data = pd.concat([
        treated.loc[[t_idx for t_idx, c_idx in matched_indices]],
        control.loc[[c_idx for t_idx, c_idx in matched_indices]]
    ])
    
    # Add match IDs
    match_ids = [f"match_{i}" for i in range(len(matched_indices))]
    matched_data['match_id'] = np.repeat(match_ids, 2)
    
    return matched_data

def check_balance(matched_data, confounders):
    """Check balance of confounders between treated and control groups."""
    balance_results = {}
    t_test_results = {}
    
    for confounder in confounders:
        treated = matched_data.loc[matched_data['DEP'] == 1, confounder]
        control = matched_data.loc[matched_data['DEP'] == 0, confounder]
        
        # Calculate standardized difference
        mean_diff = treated.mean() - control.mean()
        pooled_std = np.sqrt((treated.std()**2 + control.std()**2) / 2)
        std_diff = mean_diff / pooled_std
        
        # Perform t-test
        t_stat, p_value, _ = ttest_ind(treated, control, usevar='unequal')
        
        balance_results[confounder] = std_diff
        t_test_results[confounder] = (t_stat, p_value)
    
    return balance_results, t_test_results

def estimate_att(matched_data):
    """Estimate the Average Treatment Effect on the Treated (ATT)."""
    treated_outcome = matched_data.loc[matched_data['DEP'] == 1, 'remittances_gdp']
    control_outcome = matched_data.loc[matched_data['DEP'] == 0, 'remittances_gdp']
    return treated_outcome.mean() - control_outcome.mean()

# Main analysis pipeline
def main_analysis(df):
    # Perform propensity score matching
    matched_data = perform_propensity_matching(df)
    
    # Check balance
    confounders = ['gdp', 'inflation_gdp_deflator', 'political_stability']
    balance_results, t_test_results = check_balance(matched_data, confounders)
    
    # Print balance results
    print("\nStandardized Differences (should be < 0.1):")
    for confounder, diff in balance_results.items():
        print(f"{confounder}: {diff:.2f}")
    
    print("\nT-test Results (p-values should > 0.05):")
    for confounder, (t_stat, p_value) in t_test_results.items():
        print(f"{confounder}: p = {p_value:.3f}")
    
    # Estimate ATT
    att = estimate_att(matched_data)
    print(f"\nAverage Treatment Effect on Treated (ATT): {att:.2f}%")
    
    return matched_data, balance_results, t_test_results, att

# Execute the analysis
matched_data, balance_results, t_test_results, att = main_analysis(merged_df)


Standardized Differences (should be < 0.1):
gdp: -0.01
inflation_gdp_deflator: 0.00
political_stability: 0.02

T-test Results (p-values should > 0.05):
gdp: p = 0.882
inflation_gdp_deflator: p = 1.000
political_stability: p = 0.825

Average Treatment Effect on Treated (ATT): 1.57%


### Propensity Score Matching

The analysis used a statistical technique called propensity score matching. This method helps us compare countries with Diaspora Engagement Policies (DEPs) to countries without DEPs that are otherwise similar in terms of economic and political characteristics. By matching these countries based on their propensity scores (the likelihood of having a DEP based on their characteristics), we can create a more balanced comparison group, which helps reduce bias in our estimates.

### Balance Checks

Before comparing the remittance levels between countries with and without DEPs, we checked if the matched groups were balanced on key characteristics:

#### Standardized Differences

The standardized differences measure how different the treatment and control groups are on each characteristic. A value below 0.1 indicates good balance. Our results show:

- GDP: -0.01 (very small difference)
- Inflation (GDP deflator): 0.00 (no difference)
- Political Stability: 0.02 (very small difference)

These values indicate that the matched groups are very well balanced on these characteristics, meaning any differences we observe in remittances are less likely to be due to these factors.

#### T-tests

T-tests assess whether there's a statistically significant difference between the groups on each characteristic. A p-value above 0.05 suggests no significant difference. Our results show:

- GDP: p = 0.882 (no significant difference)
- Inflation (GDP deflator): p = 1.000 (no significant difference)
- Political Stability: p = 0.825 (no significant difference)

These results confirm that there are no statistically significant differences between the treatment and control groups on these characteristics, further supporting that our groups are well matched.

### Average Treatment Effect on the Treated (ATT)

The ATT measures the estimated effect of having a DEP on remittance inflows for the countries that actually have DEPs. Our result shows an ATT of 1.57%, meaning that countries with DEPs experience, on average, a 1.57 percentage point higher remittance inflow as a percentage of GDP compared to similar countries without DEPs.

### Interpretation

These findings suggest that Diaspora Engagement Policies (DEPs) are associated with a small but positive increase in remittance inflows relative to GDP. The balance checks confirm that this difference isn't likely due to the economic size (GDP), inflation rates, or political stability differences between countries with and without DEPs.
In simpler terms, countries that implement policies to engage their diaspora communities tend to receive slightly more money from abroad (remittances) relative to their economic size compared to similar countries without such policies. This suggests that DEPs might be effective in increasing financial contributions from diaspora communities to their home countries.
It's important to note that this is an observational study, so we can't establish causation with absolute certainty. However, the rigorous matching process and balance checks make the evidence stronger that the observed effect is indeed related to the DEPs rather than other factors.

## Step 5: Check Balance

In [63]:
# Step 5: Check balance
def calculate_standardized_difference(treated, control):
    """Calculate the standardized difference between treated and control groups."""
    mean_diff = treated.mean() - control.mean()
    pooled_std = np.sqrt((treated.std()**2 + control.std()**2) / 2)
    return mean_diff / pooled_std

def check_balance(matched_data, confounders):
    """Check balance of confounders between treated and control groups."""
    balance = {}
    t_test_results = {}
    
    for col in confounders:
        # Extract treated and control groups for this confounder
        treated = matched_data.loc[matched_data['DEP'] == 1, col].dropna()
        control = matched_data.loc[matched_data['DEP'] == 0, col].dropna()
        
        # Skip if either group has insufficient data
        if len(treated) < 2 or len(control) < 2:
            continue
        
        # Calculate standardized difference
        balance[col] = calculate_standardized_difference(treated, control)
        
        # Perform t-test and unpack all returned values
        t_stat, p_value, _ = ttest_ind(treated, control)
        t_test_results[col] = (t_stat, p_value)
    
    return balance, t_test_results

def print_balance_results(balance, t_test_results):
    """Print the balance results in a readable format."""
    # Print standardized differences
    print("Standardized Differences (should be < 0.1):")
    for col, diff in balance.items():
        print(f"{col}: {diff:.2f}")
    
    # Print t-test results
    print("\nT-test Results (p-values should > 0.05):")
    for col, (t_stat, p_value) in t_test_results.items():
        print(f"{col}: p = {p_value:.3f}")

# Execute balance checks
confounders = ['gdp', 'inflation_gdp_deflator', 'political_stability']
balance, t_test_results = check_balance(matched_data, confounders)
print_balance_results(balance, t_test_results)

Standardized Differences (should be < 0.1):
gdp: -0.01
inflation_gdp_deflator: 0.00
political_stability: 0.02

T-test Results (p-values should > 0.05):
gdp: p = 0.882
inflation_gdp_deflator: p = 1.000
political_stability: p = 0.825


### What Are Balance Checks?

After matching countries with and without Diaspora Engagement Policies (DEPs) based on their propensity scores, we need to check if the groups are balanced on important characteristics (confounders). This means we want to ensure that the countries with DEPs and those without DEPs are similar in terms of:

- GDP (economic size)
- Inflation (GDP deflator)
- Political stability

If these characteristics are balanced between the two groups, we can be more confident that any differences we observe in remittance flows are due to the DEPs rather than these other factors.

### Standardized Differences

The standardized differences measure how different the two groups are on each characteristic. A value below 0.1 indicates good balance. Our results show:

- GDP: -0.01 (very small difference)
- Inflation (GDP deflator): 0.00 (no difference)
- Political Stability: 0.02 (very small difference)

These values indicate that the matched groups are very well balanced on these characteristics. The differences are all well below the 0.1 threshold, meaning we can be confident that these factors don't systematically differ between countries with and without DEPs in our analysis.

### T-tests

T-tests assess whether there's a statistically significant difference between the groups on each characteristic. A p-value above 0.05 suggests no significant difference. Our results show:

- GDP: p = 0.882 (no significant difference)
- Inflation (GDP deflator): p = 1.000 (no significant difference)
- Political Stability: p = 0.825 (no significant difference)


These results confirm that there are no statistically significant differences between the treatment and control groups on these characteristics. This further supports that our groups are well matched and that any observed effects are more likely due to the DEPs rather than these confounding variables.

### Interpretation

The balance checks show that we've successfully created comparable groups of countries with and without DEPs. The countries in our analysis are similar in terms of their economic size, inflation rates, and political stability. This means that when we observe differences in remittance flows between these groups, we can be more confident that those differences are related to the presence of DEPs rather than being explained by these other factors.
In simpler terms, we've made sure that the countries we're comparing are similar in important ways, so any differences we find in remittances are more likely to be because of the diaspora engagement policies rather than other economic or political differences between the countries.

## Step 6: ATT

In [65]:
# Step 6: Estimate ATT
def estimate_att(matched_data):
    """Estimate the Average Treatment Effect on the Treated (ATT)."""
    treated_outcome = matched_data.loc[matched_data['DEP'] == 1, 'remittances_gdp']
    control_outcome = matched_data.loc[matched_data['DEP'] == 0, 'remittances_gdp']
    att = treated_outcome.mean() - control_outcome.mean()
    return att

# Calculate and print ATT
att = estimate_att(matched_data)
print(f"\nAverage Treatment Effect on Treated (ATT): {att:.2f}%")


Average Treatment Effect on Treated (ATT): 1.57%


### What is ATT?

The Average Treatment Effect on the Treated (ATT) is a statistical measure that tells us the estimated effect of having a Diaspora Engagement Policy (DEP) on countries that actually have such policies. It compares the average remittance inflows (as a percentage of GDP) for countries with DEPs to the average remittance inflows for similar countries without DEPs.

### Our Result

Our analysis found an ATT of 1.57%. This means that countries with DEPs experience, on average, a 1.57 percentage point higher remittance inflow relative to their GDP compared to similar countries without DEPs.

### Interpretation

In simpler terms, countries that implement policies to engage their diaspora communities tend to receive approximately 1.57% more money from abroad (relative to their economic size) compared to similar countries without such policies. This suggests that DEPs might be effective in increasing financial contributions from diaspora communities to their home countries.

### Practical Significance

While 1.57% might seem like a small number, it's important to consider the context. For countries with significant diaspora populations, even a small percentage increase in remittances can represent substantial economic value. Remittances often support household consumption, education, healthcare, and small business development in developing countries.

### Considerations

It's important to remember that this is an observational study, not a randomized experiment. While we've carefully matched countries on several characteristics and checked for balance, there could still be other factors we haven't accounted for that might influence remittance flows. Additionally, the effect size might vary across different countries and contexts.

### Conclusion

The positive ATT suggests that Diaspora Engagement Policies are associated with increased remittance inflows. This provides evidence that such policies might be beneficial for countries looking to strengthen their economic connections with diaspora communities.

## Step 7: Sensitivity Analysis

In [67]:
# Step 7: Sensitivity Analysis
def perform_mahalanobis_matching(df, confounder_cols):
    """Perform Mahalanobis distance matching and return matched dataframe."""
    from sklearn.covariance import empirical_covariance
    from scipy.spatial.distance import mahalanobis
    
    # Extract treated and control groups
    treated = df[df['DEP'] == 1]
    control = df[df['DEP'] == 0]
    
    # Extract confounder values
    confounders_treated = treated[confounder_cols].values
    confounders_control = control[confounder_cols].values
    
    # Combine confounders for covariance calculation
    confounders = np.concatenate([confounders_treated, confounders_control])
    
    # Calculate covariance matrix
    cov_matrix = empirical_covariance(confounders)
    
    # Find matches using Mahalanobis distance
    matched_indices = []
    for i, treated_row in treated.iterrows():
        treated_values = treated_row[confounder_cols].values
        distances = []
        for j, control_row in control.iterrows():
            control_values = control_row[confounder_cols].values
            distance = mahalanobis(treated_values, control_values, cov_matrix)
            distances.append((j, distance))
        # Find nearest control unit
        nearest = min(distances, key=lambda x: x[1])
        matched_indices.append((i, nearest[0]))
    
    # Create matched dataframe
    matched_data = pd.concat([
        treated.loc[[t_idx for t_idx, c_idx in matched_indices]],
        control.loc[[c_idx for t_idx, c_idx in matched_indices]]
    ])
    
    return matched_data

def estimate_att_mahalanobis(matched_data):
    """Estimate ATT using Mahalanobis matched data."""
    treated_outcome = matched_data.loc[matched_data['DEP'] == 1, 'remittances_gdp']
    control_outcome = matched_data.loc[matched_data['DEP'] == 0, 'remittances_gdp']
    return treated_outcome.mean() - control_outcome.mean()

# Perform Mahalanobis matching
confounder_cols = ['gdp', 'inflation_gdp_deflator', 'political_stability']
matched_data_mahalanobis = perform_mahalanobis_matching(matched_data, confounder_cols)

# Estimate ATT with Mahalanobis matching
att_mahalanobis = estimate_att_mahalanobis(matched_data_mahalanobis)

# Print results
print(f"\nATT with Mahalanobis Matching: {att_mahalanobis:.2f}%")


ATT with Mahalanobis Matching: 2.51%


### What is Mahalanobis Distance Matching?

Mahalanobis distance matching is a statistical technique that matches treated and control units based on their distance in multidimensional space, taking into account the correlations between variables. Unlike propensity score matching, which matches based on a single estimated probability, Mahalanobis matching considers the actual values of multiple confounding variables simultaneously. This method helps ensure that the groups are similar across all measured characteristics, not just their predicted probability of receiving treatment.

### Our Result

Using Mahalanobis distance matching, we found an ATT (Average Treatment Effect on the Treated) of 2.51%. This means that when we match countries using this different method, countries with Diaspora Engagement Policies (DEPs) experience, on average, a 2.51 percentage point higher remittance inflow relative to their GDP compared to similar countries without DEPs.

### Interpretation

The ATT from Mahalanobis matching (2.51%) is higher than the ATT from propensity score matching (1.57%). This difference suggests that the estimated effect of DEPs on remittances might vary depending on the matching method used. However, both methods show a positive effect of DEPs on remittance inflows.

### Why This Matters

Sensitivity analysis is crucial in observational studies to assess how robust the findings are to different analytical approaches. By using a different matching method, we can check if our conclusions are consistent across methods or if they depend heavily on the specific technique used.
Practical Implications
The fact that both methods show a positive effect (though of different magnitudes) suggests that DEPs are likely associated with increased remittance flows. The difference in effect size could be due to how each method handles the matching process:

- Propensity score matching relies on correctly specifying the model for the propensity scores
- Mahalanobis matching directly considers the multivariate distance between units

### Conclusion

The sensitivity analysis strengthens our confidence in the finding that DEPs are associated with increased remittance inflows. While the magnitude of the effect differs between methods, both approaches show a positive relationship. This consistency across different matching techniques suggests that the relationship is robust and less likely to be due to chance or specific methodological choices.